# 05 · SFT on the teacher's pre-labels (step 2 of the parked plan)

Trains Model A on DeepSeek's pre-labels of the 120 train agreements, then scores it and Model A
on the 5 M1 agreements. The plan is in `docs/next_steps.md`, the data in
`reports/collect_full_mix.md` and `reports/prelabel_full.md`.

1. **Data:** one example per labeled train chunk (24,000 characters, page numbers removed), plus
   empty chunks up to 30% of the set. Test agreements never enter.
2. **Short test:** 3 steps on the longest examples. It tests memory at its worst and measures the
   pace, and the run is cut to fewer steps if one epoch would pass 6.5 hours.
3. **SFT:** QLoRA on Model A, logits only for the answer, so 8,192-token examples fit a T4.
4. **Scoring:** Model A and the new adapter on the 5 M1 agreements, on the same chunks, through
   the M3 runner.

**Kaggle:** Accelerator **GPU T4 x2** (not P100), Internet **on**, Add-ons → Secrets: attach
`HF_TOKEN` with write access. "Save & Run All" runs it in the background.
**Colab:** T4 runtime, add `HF_TOKEN` under 🔑 Secrets with notebook access enabled.

**Time (estimate, not measured):** about 9 to 11 hours: up to 6.5 hours of training and 2 to 3
hours of scoring. The short test prints the real pace.

Resumable: data, checkpoints, adapters and decode records go to the private HF repos as they are
made, and a restart skips finished work. No Nebius call and no cost. All logic lives in `src/`.

In [ ]:
REPO_URL = "https://github.com/z26qin/credit-extractor.git"
BRANCH = "m4-small-loop"
INSTALL_VLLM = True      # vLLM goes into its own venv (/tmp/vllm-env); False -> transformers only


In [ ]:
import os, subprocess, sys
ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else (
    "/content" if os.path.exists("/content") else os.getcwd())
REPO = os.path.join(ROOT, "credit-extractor")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only"], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
# register the repo on sys.path permanently, so `import src` survives a kernel restart
import site
with open(os.path.join(site.getsitepackages()[0], "credit_extractor.pth"), "w") as f:
    f.write(REPO + "\n")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

pip = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(pip + ["-r", "requirements.txt"], check=True)

# 1) platform Python (transformers backend): never install vLLM here. Its torch pin
#    breaks the preinstalled torchaudio, which transformers imports.
BASE_CHECK = ("import torch, transformers\n"
              "from transformers import Qwen3ForCausalLM\n"
              "print('base: torch', torch.__version__, 'cuda', torch.version.cuda,"
              " 'gpu', torch.cuda.is_available(), '| transformers', transformers.__version__)")
r = subprocess.run([sys.executable, "-c", BASE_CHECK], capture_output=True, text=True)
if r.returncode and "torchaudio" in r.stderr.lower():
    print("torchaudio does not match torch; removing it (not needed for text models)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"])
    r = subprocess.run([sys.executable, "-c", BASE_CHECK], capture_output=True, text=True)
print(r.stdout or r.stderr[-1500:])
assert r.returncode == 0, "base env broken: Run -> Factory reset, then run all cells again"

# 2) vLLM in its own venv; uv picks the torch build that matches this GPU driver
VENV = "/tmp/vllm-env"; VPY = VENV + "/bin/python"
VLLM_CHECK = ("import torch, vllm\n"
              "print('vllm venv: vllm', vllm.__version__, '| torch', torch.__version__,"
              " 'cuda', torch.version.cuda, 'gpu', torch.cuda.is_available())")
if INSTALL_VLLM and not os.path.exists(VPY):
    subprocess.run(pip + ["uv"], check=True)
    uv = [sys.executable, "-m", "uv"]
    subprocess.run(uv + ["venv", "-q", VENV, "--python", sys.executable], check=True)
    r = subprocess.run(uv + ["pip", "install", "-q", "--python", VPY, "--torch-backend=auto",
                             "-r", "requirements-gpu.txt", "-r", "requirements.txt", "wrapt"],
                       capture_output=True, text=True)
    if r.returncode:
        print("vllm install FAILED -> transformers only\n" + r.stderr[-2000:])
        import shutil; shutil.rmtree(VENV, ignore_errors=True)
if os.path.exists(VPY):
    r = subprocess.run([VPY, "-c", VLLM_CHECK], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else
          "vllm venv not usable (runner will fall back to transformers):\n" + r.stderr[-1500:])
else:
    print("no vllm venv: runner uses transformers")

# 3) training in its own venv, with the versions the CPU tests ran on
TVENV = "/tmp/train-env"; TPY = TVENV + "/bin/python"
TRAIN_CHECK = ("import torch, transformers, peft, trl, bitsandbytes\n"
               "print('train venv: torch', torch.__version__, 'cuda', torch.version.cuda,"
               " 'gpu', torch.cuda.is_available(), '| transformers', transformers.__version__,"
               " '| peft', peft.__version__, '| trl', trl.__version__)")
if not os.path.exists(TPY):
    subprocess.run(pip + ["uv"], check=True)
    uv = [sys.executable, "-m", "uv"]
    subprocess.run(uv + ["venv", "-q", TVENV, "--python", "3.12"], check=True)
    r = subprocess.run(uv + ["pip", "install", "-q", "--python", TPY, "--torch-backend=auto",
                             "-r", "requirements-train.txt", "-r", "requirements.txt"],
                       capture_output=True, text=True)
    assert r.returncode == 0, "training install failed:\n" + r.stderr[-2000:]
r = subprocess.run([TPY, "-c", TRAIN_CHECK], capture_output=True, text=True)
print(r.stdout or r.stderr[-1500:])
assert r.returncode == 0, "training venv not usable"


In [ ]:
# after a kernel restart, start again from this cell (the cells below need `env`)
from src.platform import setup
env = setup()
assert env.data_repo, "HF_TOKEN missing: attach it as a secret (outputs must sync to the Hub)"
assert env.gpus, "no GPU visible: pick GPU T4 x2 (Kaggle) or a T4 runtime (Colab)"


In [ ]:
from src import real_loop
cfg = real_loop.load_cfg()
data, info = real_loop.build_data(env, cfg)      # same seed, same examples after a restart
info

In [ ]:
# memory at the longest examples, seconds per step, and how many steps fit
short = real_loop.short_test(env, cfg, data, info)
short

In [ ]:
adapter = real_loop.train(env, cfg, data, short["max_steps"])

In [ ]:
# Model A first, then the new adapter, both on the same 24,000-character chunks
results = real_loop.evaluate(env, cfg, adapter)

In [ ]:
report = real_loop.write_report(env, cfg, info, short, adapter, results)
print(f"session wall time: {env.session_minutes():.1f} min")